<a href="https://colab.research.google.com/github/BigFoots625/IntroductionMachineLearningwithPython/blob/main/Chapter2_SupervisedLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  **Supervised Learning**
Supervised learning involves learning a link between two datasets: the observed data (inputs) and an external variable that we are trying to predict (the target).

In this chapter, we will use a few standard datasets:
* The **Forge dataset** (a synthetic dataset for binary classification).
* The **Wave dataset** (a synthetic dataset for regression).
* The **Breast Cancer dataset** (a real-world, high-dimensional dataset for classification).

In [1]:
!pip install mglearn
import mglearn
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.4/581.4 kB 21.2 MB/s eta 0:00:00


In [2]:
from sklearn.datasets import load_breast_cancer

# Load the breast cancer dataset
cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, stratify=cancer.target, random_state=42)

### **Theoretical Explanation: Linear Models for Regression**
Linear models make predictions using a linear function of the input features:
$$\hat{y} = w_0 x_0 + w_1 x_1 + \dots + w_p x_p + b$$
Here, $x$ represents the features, $w$ represents the learned weights (coefficients), and $b$ is the bias (intercept).

Standard linear regression (Ordinary Least Squares) simply minimizes the mean squared error between predictions and the true targets. However, this often leads to **overfitting** on high-dimensional data. To fix this, we use regularization:

1. **Ridge Regression (L2 Regularization):** Adds a penalty proportional to the *square* of the magnitude of the coefficients. It shrinks all weights toward zero but rarely exactly to zero.
2. **Lasso Regression (L1 Regularization):** Adds a penalty proportional to the *absolute value* of the coefficients. This acts as automatic feature selection, as it forces some coefficients to become exactly zero, making the model easier to interpret.

In [3]:
from sklearn.linear_model import Ridge, Lasso

# Ridge Regression (L2)
ridge = Ridge(alpha=1.0).fit(X_train, y_train)
print(f"Ridge Training score: {ridge.score(X_train, y_train):.3f}")
print(f"Ridge Test score: {ridge.score(X_test, y_test):.3f}")

# Lasso Regression (L1)
lasso = Lasso(alpha=0.01, max_iter=100000).fit(X_train, y_train)
print(f"Lasso Training score: {lasso.score(X_train, y_train):.3f}")
print(f"Lasso Test score: {lasso.score(X_test, y_test):.3f}")
print(f"Number of features used by Lasso: {np.sum(lasso.coef_ != 0)}")

Ridge Training score: 0.747
Ridge Test score: 0.733
Lasso Training score: 0.694
Lasso Test score: 0.697
Number of features used by Lasso: 8


### **Theoretical Explanation: Linear Models for Classification**
For classification, linear models learn a decision boundary (a line, plane, or hyperplane) that separates the classes. The two most common linear classification algorithms are:
1. **Logistic Regression:** Despite the name, this is a classification algorithm. It models the probability of a data point belonging to a certain class using a logistic function.
2. **Linear Support Vector Classifiers (LinearSVC):** This model attempts to find the hyperplane that maximizes the margin (distance) between the closest data points of different classes.

Both models use a regularization parameter $C$. A lower $C$ increases regularization (simpler model), while a higher $C$ decreases regularization (model tries to perfectly fit the training data).

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

# Logistic Regression
logreg = LogisticRegression(max_iter=10000).fit(X_train, y_train)
print(f"Logistic Regression Test score: {logreg.score(X_test, y_test):.3f}")

# Linear SVC
# (Note: LinearSVC requires scaled data for optimal performance, but we use raw data here for demonstration)
linearsvc = LinearSVC(max_iter=10000, dual="auto").fit(X_train, y_train)
print(f"LinearSVC Test score: {linearsvc.score(X_test, y_test):.3f}")

Logistic Regression Test score: 0.958
LinearSVC Test score: 0.958


### **Theoretical Explanation: Naive Bayes Classifiers**
Naive Bayes classifiers are incredibly fast and train by gathering simple per-class statistics from each feature individually. They are called "naive" because they assume that all features are entirely independent of one another—a mathematically strict assumption that is rarely true in real life.

Despite this, models like `GaussianNB` (for continuous data) or `MultinomialNB` (for discrete count data, like text) are highly effective baselines, particularly for very high-dimensional datasets.

In [5]:
from sklearn.naive_bayes import GaussianNB

# Instantiate and train Gaussian Naive Bayes
gnb = GaussianNB().fit(X_train, y_train)
print(f"GaussianNB Training score: {gnb.score(X_train, y_train):.3f}")
print(f"GaussianNB Test score: {gnb.score(X_test, y_test):.3f}")

GaussianNB Training score: 0.946
GaussianNB Test score: 0.937


### **Theoretical Explanation: Decision Trees and Ensembles**
**Decision Trees** split the data by asking a series of if/else questions. They are highly interpretable but prone to overfitting. We mitigate this using **Ensembles**:

1. **Random Forests:** Builds many independent decision trees on bootstrapped (randomly sampled) subsets of the data and averages their predictions. This reduces variance without increasing bias.
2. **Gradient Boosted Regression Trees (GBRT):** Builds trees sequentially. Each new tree in the sequence specifically attempts to correct the residual errors made by the previous trees. It uses a `learning_rate` parameter to control how strongly each tree tries to correct the mistakes of the preceding trees.

In [6]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Random Forest
forest = RandomForestClassifier(n_estimators=100, random_state=0).fit(X_train, y_train)
print(f"Random Forest Test score: {forest.score(X_test, y_test):.3f}")

# Gradient Boosting
gbrt = GradientBoostingClassifier(random_state=0).fit(X_train, y_train)
print(f"Gradient Boosting Test score: {gbrt.score(X_test, y_test):.3f}")

Random Forest Test score: 0.958
Gradient Boosting Test score: 0.958


### **Theoretical Explanation: Kernelized Support Vector Machines**
Linear models fail on datasets that are not linearly separable. Kernelized SVMs use the **kernel trick** to implicitly map the input data into a much higher-dimensional space where a linear hyperplane *can* separate the classes.

The most common is the Radial Basis Function (RBF) kernel:
$$k(x_1, x_2) = \exp(-\gamma ||x_1 - x_2||^2)$$
The parameter $\gamma$ controls the width of the Gaussian kernel (how far the influence of a single training example reaches), while $C$ controls the regularization.

In [7]:
from sklearn.svm import SVC

svm = SVC(C=1.0, gamma='scale').fit(X_train, y_train)
print(f"SVM Test score: {svm.score(X_test, y_test):.3f}")

SVM Test score: 0.923


### **Theoretical Explanation: Neural Networks (Multilayer Perceptrons)**
Multilayer Perceptrons (MLPs) consist of an input layer, one or more hidden layers, and an output layer. Each connection has a weight. To allow the network to learn non-linear patterns, a non-linear activation function (like ReLU or `tanh`) is applied to the output of the hidden layers.
$$f(x) = \max(0, x)$$
Neural networks are powerful but require careful tuning and data scaling (features should ideally have a mean of 0 and a variance of 1).

In [8]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(random_state=42, max_iter=1000).fit(X_train, y_train)
print(f"Neural Network Test score: {mlp.score(X_test, y_test):.3f}")

Neural Network Test score: 0.916


### **Theoretical Explanation: Uncertainty Estimates**
Scikit-learn classifiers often provide two methods to quantify their confidence in a prediction:
1. `decision_function`: Returns a floating-point value for each data point. The sign indicates the predicted class, and the magnitude indicates the confidence (distance from the decision boundary).
2. `predict_proba`: Returns the probability (a value between 0 and 1) of the data point belonging to each class. The probabilities for all classes sum to 1.

In [9]:
# Demonstrating uncertainty estimates using our previously trained Gradient Boosting model
print("Decision function shape:", gbrt.decision_function(X_test).shape)
print("Decision function (first 5):\n", gbrt.decision_function(X_test)[:5])

print("\nPredict proba shape:", gbrt.predict_proba(X_test).shape)
print("Predict proba (first 5):\n", gbrt.predict_proba(X_test)[:5])

Decision function shape: (143,)
Decision function (first 5):
 [ 7.45385091 -7.70537895  1.73189138  4.77360828  3.61840918]

Predict proba shape: (143, 2)
Predict proba (first 5):
 [[5.78871549e-04 9.99421128e-01]
 [9.99549805e-01 4.50195220e-04]
 [1.50345811e-01 8.49654189e-01]
 [8.37903423e-03 9.91620966e-01]
 [2.61245184e-02 9.73875482e-01]]


### **Chapter 2 Summary**
* **Objective:** We explored the theoretical foundations and practical implementations of the core supervised learning algorithms.
* **Key Concepts:** We covered regularization (L1/L2), hyperparameter tuning ($C$, $\gamma$, `alpha`), non-linear transformations via kernels and hidden layers, and how models calculate prediction confidence.
* **Takeaway:** * **Linear Models & Naive Bayes:** Fast, baseline models ideal for very large or sparse data.
    * **Trees & Ensembles:** Extremely powerful out-of-the-box (especially Random Forests and GBRT), robust to unscaled data.
    * **SVMs & Neural Networks:** State-of-the-art performance for complex relationships, but highly sensitive to data scaling and require significant parameter tuning.